# Decision identity on synthetic data and test images

**Computes:** Checks, for eight assignment rules, whether class-uniform and class-wise reducers of the interval change the type-1 decision, and measures the overlap band.

**Inputs:** none (synthetic data and scikit-image test images).  
**Outputs:** `results/01/results_01.json` (the report is printed in the notebook).


In [ ]:
import os, numpy as np, pandas as pd, json, time, platform
OUTDIR = os.path.abspath(os.path.join(os.environ.get("RESULTS_DIR", "results"), "01")); os.makedirs(OUTDIR, exist_ok=True)
import skimage
from skimage import data, transform
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__, "| skimage", skimage.__version__)

In [ ]:
import numpy as np

def _softmax_neglog(D, p, logw=None):
    z = -p * np.log(D)
    if logw is not None:
        z = z + logw
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def fcm(D, m):
    return _softmax_neglog(D, 2.0 / (m - 1.0))

def pcm(D, m, eta=0.09):
    return 1.0 / (1.0 + (D**2 / eta) ** (1.0 / (m - 1.0)))

def prior_fcm(D, m, priors):
    return _softmax_neglog(D, 2.0 / (m - 1.0), logw=np.log(priors)[None, :])

def gauss_shared(D, s):
    return np.exp(-D**2 / (2.0 * s**2))

def gauss_class(D, t, base):
    sig = base[None, :] + t
    return np.exp(-D**2 / (2.0 * sig**2))

def distances(X, P, kind, C):
    diff = X[:, None, :] - P[None, :, :]
    e = np.sqrt((diff**2).sum(axis=2))
    if kind == "euclid":
        D = e
    elif kind == "kernel":
        D = np.sqrt(2.0 * (1.0 - np.exp(-e**2 / (2 * 0.35**2))))
    elif kind == "metric":
        a = np.array([1.0, 4.0] * C)[:C]
        D = np.sqrt(a)[None, :] * e
    elif kind == "spatial":
        o = np.array([0.02, 0.05] * C)[:C]
        D = e + o[None, :]
    else:
        raise ValueError(kind)
    return np.maximum(D, 1e-12)

FUZZ_PAIRS = [(a, b) for a in (1.1, 1.3, 1.5, 2.0) for b in (3.0, 5.0, 10.1)]
SIGMA_PAIRS = [(a, b) for a in (0.05, 0.10, 0.20, 0.30) for b in (0.40, 0.60, 1.00)]
T_PAIRS = [(a, b) for a in (0.00, 0.02, 0.05, 0.10) for b in (0.20, 0.40, 0.80)]

def rules():
    return {
        "fcm":            (lambda D, th, C: fcm(D, th), "euclid", FUZZ_PAIRS, (1, 1, 1)),
        "kernel_fcm":     (lambda D, th, C: fcm(D, th), "kernel", FUZZ_PAIRS, (1, 1, 1)),
        "metric_fcm":     (lambda D, th, C: fcm(D, th), "metric", FUZZ_PAIRS, (1, 1, 1)),
        "spatial_fcm":    (lambda D, th, C: fcm(D, th), "spatial", FUZZ_PAIRS, (1, 1, 1)),
        "pcm":            (lambda D, th, C: pcm(D, th), "euclid", FUZZ_PAIRS, (0, 1, 1)),
        "gauss_shared":   (lambda D, th, C: gauss_shared(D, th), "euclid", SIGMA_PAIRS, (0, 1, 1)),
        "prior_fcm":      (lambda D, th, C: prior_fcm(D, th, np.arange(1, C + 1) / np.arange(1, C + 1).sum()),
                           "euclid", FUZZ_PAIRS, (1, 0, 0)),
        "gauss_class":    (lambda D, th, C: gauss_class(D, th, np.linspace(0.05, 0.30, C)),
                           "euclid", T_PAIRS, (0, 0, 0)),
    }

def analyse(Us, D, rng, tie_rel=1e-9):
    L = np.minimum.reduce(Us)
    U = np.maximum.reduce(Us)
    N, C = D.shape
    Ds = np.sort(D, axis=1)
    ok_geom = (Ds[:, 1] - Ds[:, 0]) > tie_rel * Ds[:, 1]
    strict = np.logical_and.reduce([np.sort(u, axis=1)[:, -1] > np.sort(u, axis=1)[:, -2] for u in Us])
    ok = ok_geom & strict
    t1 = [u.argmax(axis=1) for u in Us]
    k = t1[0]
    idx = np.arange(N)

    def disagree(red):
        dec = red.argmax(axis=1)
        return np.logical_and.reduce([dec != t for t in t1])

    alpha_shared = rng.uniform(size=(N, 1))
    alpha_class = rng.uniform(size=(N, C))
    reducers = {
        "lower": L,
        "upper": U,
        "midpoint": 0.5 * (L + U),
        "shared_alpha": alpha_shared * L + (1 - alpha_shared) * U,
        "class_alpha": alpha_class * L + (1 - alpha_class) * U,
    }
    adv = U.copy()
    adv[idx, k] = L[idx, k]
    reducers["adversarial"] = adv

    Uo = U.copy()
    Uo[idx, k] = -np.inf
    O = Uo.max(axis=1) > L[idx, k]

    out = {f"dis_{r}": disagree(v)[ok].mean() for r, v in reducers.items()}
    out.update({f"dis1_{r}": (v.argmax(axis=1) != k)[ok].mean() for r, v in reducers.items()})
    out["float_tie_excluded"] = float((ok_geom & ~strict).mean())
    out["O"] = O[ok].mean()
    flips_adv = (reducers["adversarial"].argmax(axis=1) != k)
    flips_cls = (reducers["class_alpha"].argmax(axis=1) != k)
    out["adv_equals_O"] = int(((flips_adv != O) & ok).sum())
    out["class_outside_O"] = int((flips_cls & ~O & ok).sum())
    out["t1_consistent"] = float(np.mean([np.mean((t == k)[ok]) for t in t1]))
    w = (U - L)[idx, k]
    near = ok & (Ds[:, 1] / Ds[:, 0] < 1.02)
    out["w_near_tie"] = float(np.median(w[near])) if near.any() else np.nan
    out["w_max"] = float(w[ok].max())
    return out

In [ ]:
# Synthetic configurations: 8 rules, C in {2,3,4,5,8}, dimension 1-3, 10 prototype sets, 12 parameter pairs.
t0 = time.time()
CS, DIMS, SEEDS, N = (2, 3, 4, 5, 8), (1, 2, 3), 10, 8000
syn = []
for name, (f, kind, pairs, ax) in rules().items():
    for Cn in CS:
        for dim in DIMS:
            for seed in range(SEEDS):
                rng = np.random.default_rng(1000 * Cn + 100 * dim + seed)
                P = rng.uniform(size=(Cn, dim)); X = rng.uniform(size=(N, dim))
                D = distances(X, P, kind, Cn)
                for (a, b) in pairs:
                    o = analyse([f(D, a, Cn), f(D, b, Cn)], D, rng)
                    o.update(rule=name, C=Cn, dim=dim, seed=seed, pair=f"{a}-{b}", P1=ax[0], P2=ax[1], P3=ax[2])
                    syn.append(o)
syn = pd.DataFrame(syn)
print(f"{len(syn)} synthetic configurations in {time.time()-t0:.1f}s")

In [ ]:
ORDER = ["fcm", "kernel_fcm", "metric_fcm", "spatial_fcm", "pcm", "gauss_shared", "prior_fcm", "gauss_class"]
syn_tab = syn.groupby(["rule", "C"]).agg(
    type1_consistent=("t1_consistent", "min"), float_ties_excl=("float_tie_excluded", "max"),
    lower=("dis_lower", "max"), upper=("dis_upper", "max"), midpoint=("dis_midpoint", "max"),
    shared_w=("dis_shared_alpha", "max"), class_w=("dis_class_alpha", "max"),
    O_median=("O", "median"), O_min=("O", "min"), O_max=("O", "max"),
    adv_vs_O_mismatch=("adv_equals_O", "sum"), class_flips_outside_O=("class_outside_O", "sum")).reindex(ORDER, level=0)
print(syn_tab.round(4).to_string())

In [ ]:
# Natural test images: FCM prototypes held fixed, then the fuzzifier-pair interval.
def load_images():
    ims = {"camera": data.camera()[..., None] / 255.0, "astronaut": data.astronaut() / 255.0,
           "coffee": data.coffee() / 255.0, "chelsea": data.chelsea() / 255.0}
    out = {}
    for k, v in ims.items():
        h, w = v.shape[:2]; s = 256 / max(h, w)
        out[k] = transform.resize(v, (int(h * s), int(w * s)), anti_aliasing=True).reshape(-1, v.shape[2])
    return out

def fcm_prototypes(X, Cn, m=2.0, iters=200, seed=0):
    rng = np.random.default_rng(seed)
    P = X[rng.choice(len(X), Cn, replace=False)]
    for _ in range(iters):
        D = np.maximum(np.linalg.norm(X[:, None] - P[None], axis=2), 1e-12)
        U = fcm(D, m) ** m
        Pn = (U.T @ X) / U.sum(0)[:, None]
        if np.abs(Pn - P).max() < 1e-9: break
        P = Pn
    return P

t0 = time.time(); img = []
for name, X in load_images().items():
    for Cn in (2, 3, 4, 5):
        P = fcm_prototypes(X, Cn)
        D = np.maximum(np.linalg.norm(X[:, None] - P[None], axis=2), 1e-12)
        Ds = np.sort(D, 1); ratio = Ds[:, 1] / Ds[:, 0]
        sets = [((a, b), [fcm(D, a), fcm(D, b)]) for (a, b) in FUZZ_PAIRS]
        sets.append(("all7", [fcm(D, m) for m in (1.1, 1.3, 1.5, 2.0, 3.0, 5.0, 10.1)]))
        for pair, Us in sets:
            o = analyse(Us, D, np.random.default_rng(7))
            L, U = np.minimum.reduce(Us), np.maximum.reduce(Us); k = Us[0].argmax(1); idx = np.arange(len(X))
            Uo = U.copy(); Uo[idx, k] = -np.inf; O = Uo.max(1) > L[idx, k]
            o.update(image=name, C=Cn, pair=str(pair), N=len(X),
                     ratio_in_O=float(np.median(ratio[O])) if O.any() else np.nan,
                     ratio_out_O=float(np.median(ratio[~O])))
            img.append(o)
img = pd.DataFrame(img)
print(f"{len(img)} image configurations in {time.time()-t0:.1f}s")

In [ ]:
pairs = img[img.pair != "all7"]
img_tab = pairs.groupby(["image", "C"]).agg(N=("N", "first"),
    lower=("dis_lower", "max"), upper=("dis_upper", "max"), midpoint=("dis_midpoint", "max"),
    shared_w=("dis_shared_alpha", "max"), class_w=("dis_class_alpha", "max"),
    O_median=("O", "median"), O_min=("O", "min"), O_max=("O", "max"),
    adv_vs_O_mismatch=("adv_equals_O", "sum"), class_flips_outside_O=("class_outside_O", "sum"),
    ratio_in_O=("ratio_in_O", "median"), ratio_out_O=("ratio_out_O", "median"),
    w_near_tie=("w_near_tie", "median"), w_max=("w_max", "median"))
print(img_tab.round(4).to_string())
all7 = img[img.pair == "all7"][["image", "C", "dis_lower", "dis_upper", "dis_midpoint", "dis_shared_alpha", "O", "adv_equals_O"]]
print(all7.round(4).to_string(index=False))

In [ ]:
# Report.
P3_RULES = ["fcm", "kernel_fcm", "metric_fcm", "spatial_fcm", "pcm", "gauss_shared"]
P1P3_RULES = ["fcm", "kernel_fcm", "metric_fcm", "spatial_fcm"]
SYM = ["dis_lower", "dis_upper", "dis_midpoint", "dis_shared_alpha"]
FAITH = SYM + ["dis_class_alpha", "dis_adversarial"]
s3 = syn[syn.rule.isin(P3_RULES)]; s13 = syn[syn.rule.isin(P1P3_RULES)]; noP3 = syn[~syn.rule.isin(P3_RULES)]
checks = [
 ("(a) synthetic: class-uniform reducers, (P3) rules, all C", s3[SYM].to_numpy().max(), "== 0"),
 ("(a) images: class-uniform reducers, all C, pairs and all7", img[SYM].to_numpy().max(), "== 0"),
 ("(c) synthetic: C=2 with (P1)+(P3): size of O", s13[s13.C == 2]["O"].max(), "== 0"),
 ("(c) synthetic: C=2 with (P1)+(P3): any faithful reducer", s13[s13.C == 2][FAITH].to_numpy().max(), "== 0"),
 ("(c) images: C=2: size of O", img[img.C == 2]["O"].max(), "== 0"),
 ("(b) adversarial flips differ from O (points, all runs)", int(syn.adv_equals_O.sum() + img.adv_equals_O.sum()), "== 0"),
 ("(b) class-weighted flips outside O (points, all runs)", int(syn.class_outside_O.sum() + img.class_outside_O.sum()), "== 0"),
]
allpass = True
for label, val, crit in checks:
    ok = (val == 0); allpass &= bool(ok)
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}: {val} (expected {crit})")
print("  ALL IDENTITY CHECKS PASS" if allpass else "  *** SOME CHECK FAILED ***")

def rng_str(x): return f"{100*np.min(x):.1f}-{100*np.max(x):.1f}%"
fam = syn[syn.rule.isin(P1P3_RULES)].groupby(["rule", "C"]).O.median()
print(f"  O, FCM family, C=3: {rng_str(fam.xs(3, level='C'))} | C=8: {rng_str(fam.xs(8, level='C'))}")
print(f"  O at C=2 without (P1): possibilistic {100*syn[(syn.rule=='pcm')&(syn.C==2)].O.median():.1f}% | shared-width Gaussian {100*syn[(syn.rule=='gauss_shared')&(syn.C==2)].O.median():.1f}%")
print(f"  (P3) violated, midpoint reducer, max disagreement: prior-weighted {100*syn[syn.rule=='prior_fcm'].dis_midpoint.max():.1f}% | class-specific Gaussian {100*syn[syn.rule=='gauss_class'].dis_midpoint.max():.1f}%")
io = pairs[pairs.C >= 3].groupby(["image", "C"]).O.median()
print(f"  images, C=3-5: O median {rng_str(io)}; d2/d1 inside O {pairs[pairs.C>=3].ratio_in_O.median():.3f} "
      f"[{pairs[pairs.C>=3].groupby(['image','C']).ratio_in_O.median().min():.3f}, {pairs[pairs.C>=3].groupby(['image','C']).ratio_in_O.median().max():.3f}] "
      f"vs outside {pairs[pairs.C>=3].groupby(['image','C']).ratio_out_O.median().min():.2f}-{pairs[pairs.C>=3].groupby(['image','C']).ratio_out_O.median().max():.2f}")
print(f"  possibilistic points excluded for float64 ties (max over runs): {100*syn[syn.rule=='pcm'].float_tie_excluded.max():.1f}%")
print(f"  configurations: synthetic {len(syn)}, images {len(img)}")

summary = {"all_identity_checks_pass": allpass, "checks": {c[0]: float(c[1]) for c in checks},
           "syn_table": syn_tab.round(6).reset_index().to_dict(orient="records"),
           "img_table": img_tab.round(6).reset_index().to_dict(orient="records")}
json.dump(summary, open(os.path.join(OUTDIR, "results_01.json"), "w"), indent=1, default=float)